# Трек B: датасет, обучение ft_yolo_v1 и детекции
Вход: данные соревнования, датасеты `ts-code` и `ts-models`. GPU включён, интернет выключен. Запуск: Save Version (Run All).

Возобновление после обрыва: подключите вывод прошлой версии этого ноутбука как входные данные и допишите к команде обучения `--resume <путь>/artifacts/models/ft_yolo_v1/weights/last.pt`.

In [ ]:
# Подготовка: код из датасета ts-code, данные соревнования, веса и колёса из ts-models. Интернет выключен.
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

INPUT = Path('/kaggle/input')
code = [p.parent for p in INPUT.rglob('run.py') if (p.parent / 'ts' / '__init__.py').is_file()]
data = [p.parent for p in INPUT.rglob('train_labels.csv')
        if (p.parent / 'tables.csv').is_file() and (p.parent / 'data_card.json').is_file()]
models = [p.parent for p in INPUT.rglob('yolo11x.pt')]
assert len(code) == 1, f'Подключите датасет ts-code (найдено {len(code)})'
assert len(data) == 1, f'Подключите данные соревнования (найдено {len(data)})'
assert len(models) == 1, f'Подключите датасет ts-models (найдено {len(models)})'
REPO, MODELS = Path('/kaggle/working/table-state'), models[0]
shutil.copytree(code[0], REPO, dirs_exist_ok=True)
os.environ['TS_DATA'] = str(data[0])
os.environ['TS_ARTIFACTS'] = '/kaggle/working/artifacts'
os.environ['YOLO_OFFLINE'] = 'true'
os.chdir(REPO)

# Колёса ставим без сети и без зависимостей: torch, torchvision и opencv уже есть в образе.
wheels = sorted((MODELS / 'wheels').glob('*.whl'))
need = [w for w in wheels if not (w.name.startswith('nvidia_ml_py') and importlib.util.find_spec('pynvml'))]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', *map(str, need)], check=True)
import torch, ultralytics
print('код:', (REPO / 'GIT_SHA').read_text().strip(), '| данные:', data[0], '| веса:', MODELS)
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| GPU:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ, включите GPU в настройках')


In [ ]:
!python tracks/det/build_dataset.py --name ft_data_v1 --classes v1 --pseudo-step 2

In [ ]:
!python tracks/det/train.py --config tracks/det/configs/ft_yolo_v1.yaml --data /kaggle/working/artifacts/datasets/ft_data_v1 --weights {MODELS}/yolo11m.pt --hours 2.7

In [ ]:
!python tracks/det/infer.py --source ft_yolo --weights /kaggle/working/artifacts/models/ft_yolo_v1/ft_yolo_v1.pt --half

In [ ]:
!cd /kaggle/working/artifacts && zip -q -r datasets_ft_data_v1.zip datasets/ft_data_v1 && rm -rf datasets /kaggle/working/table-state && ls -la . dets models/ft_yolo_v1